# Evaluation Metric Functions
the main metrics we will be evaluating based on are Recall@K, NDCG@10 and Hit Rate@K. 

In [ ]:
import pandas as pd 

test_df = pd.read_csv("../data/processed/test.csv")
test_positive = test_df[test_df["rating"] >= 4]

user_id = test_positive["user_id"].iloc[0]

relevant = set(
    test_positive[
        test_positive["user_id"] == user_id
    ]["gmap_id"]
)


In [4]:
def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = len(set(recommended_k) & set(relevant))

    return hits / len(relevant)

Recall@K = how many items in the top K that the user will actually like.

In [28]:
# test for recall
recommended = ["A", "B", "C", "D", "E"]
relevant = {"B", "E"}

print(recall_at_k(recommended, relevant, 3))

recommended = ["A", "B", "C", "D", "E"]
relevant = {"B", "E"}

print(recall_at_k(recommended, relevant, 5))

recommended = ["A", "B", "C"]
relevant = {"X", "Y"}

print(recall_at_k(recommended, relevant, 3))

0.5
1.0
0.0


In [19]:
import math

def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    # Actual DCG
    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)

    # Best possible DCG
    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg

NDCG@K = How well ranked the relevant activity was ranked (are more relevant closer to the top?)

In [ ]:
# testing for ndgc 
recommended = ["A", "B", "C", "D", "E"]
relevant = {"A", "B"}

print(ndcg_at_k(recommended, relevant, 5))

recommended = ["C", "D", "A", "E", "B"]
relevant = {"A", "B"}

print(ndcg_at_k(recommended, relevant, 5))

1.0
0.5437713091520254


In [23]:
def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = set(recommended_k) & set(relevant)

    if len(hits) > 0:
        return 1
    else:
        return 0

Hit Rate@K: Was at least one relevant item put in the top K? 

In [26]:
# testing for hit rate
recommended = ["A", "B", "C", "D", "E"]
relevant = {"B", "X"}

print(hit_rate_at_k(recommended, relevant, 5))

recommended = ["A", "B", "C", "D", "E"]
relevant = {"X", "Y"}

print(hit_rate_at_k(recommended, relevant, 5))

1
0


In [ ]:
def calculate_metrics(recommended, relevant, k=10):

    recall = recall_at_k(recommended, relevant, k)
    hit_rate = hit_rate_at_k(recommended, relevant, k)
    ndcg = ndcg_at_k(recommended, relevant, k)

    return {
        f"Recall@{k}": recall,
        f"HitRate@{k}": hit_rate,
        f"NDCG@{k}": ndcg
    }